<a href="https://colab.research.google.com/github/AntonKrokhin/a-b-testing-tool/blob/main/A_B_Testing_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# Connecting to Google BigQuery
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
project_id = "data-analytics-mate"
client = bigquery.Client(project = "data-analytics-mate")

In [13]:
# Writing an SQL query
query = """
WITH session_info AS(
 SELECT
       s.date,
       s.ga_session_id,
       sp.country,
       sp.device,
       sp.continent,
       sp.channel,
       ab.test,
       ab.test_group
 FROM `DA.ab_test` ab
 JOIN `DA.session` s
 ON ab.ga_session_id = s.ga_session_id
 JOIN `DA.session_params` sp
 ON ab.ga_session_id = sp.ga_session_id
),
session_with_orders AS(
 SELECT
       session_info.date,
       session_info.country,
       session_info.device,
       session_info.continent,
       session_info.channel,
       session_info.test,
       session_info.test_group,
       COUNT(DISTINCT o.ga_session_id) AS session_with_orders
 FROM `DA.order` o
 JOIN session_info
 ON o.ga_session_id = session_info.ga_session_id
 GROUP BY
       session_info.date,
       session_info.country,
       session_info.device,
       session_info.continent,
       session_info.channel,
       session_info.test,
       session_info.test_group
),
events AS(
 SELECT
       session_info.date,
       session_info.country,
       session_info.device,
       session_info.continent,
       session_info.channel,
       session_info.test,
       session_info.test_group,
       ep.event_name,
       COUNT(ep.ga_session_id) AS event_cnt
 FROM `DA.event_params` ep
 JOIN session_info
 ON ep.ga_session_id = session_info.ga_session_id
 GROUP BY
       session_info.date,
       session_info.country,
       session_info.device,
       session_info.continent,
       session_info.channel,
       session_info.test,
       session_info.test_group,
       ep.event_name
),
session AS(
 SELECT
       session_info.date,
       session_info.country,
       session_info.device,
       session_info.continent,
       session_info.channel,
       session_info.test,
       session_info.test_group,
       COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
 FROM session_info
 GROUP BY
       session_info.date,
       session_info.country,
       session_info.device,
       session_info.continent,
       session_info.channel,
       session_info.test,
       session_info.test_group
),
account AS(
 SELECT
       session_info.date,
       session_info.country,
       session_info.device,
       session_info.continent,
       session_info.channel,
       session_info.test,
       session_info.test_group,
       COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
 FROM `DA.account_session` acs
 JOIN session_info
 ON acs.ga_session_id = session_info.ga_session_id
 GROUP BY
       session_info.date,
       session_info.country,
       session_info.device,
       session_info.continent,
       session_info.channel,
       session_info.test,
       session_info.test_group
)


SELECT
     session_with_orders.date,
     session_with_orders.country,
     session_with_orders.device,
     session_with_orders.continent,
     session_with_orders.channel,
     session_with_orders.test,
     session_with_orders.test_group,
     'session with orders' AS event_name,
     session_with_orders.session_with_orders AS value
FROM session_with_orders
UNION ALL
SELECT
     events.date,
     events.country,
     events.device,
     events.continent,
     events.channel,
     events.test,
     events.test_group,
     event_name,
     event_cnt AS value
FROM events
UNION ALL
SELECT
     session.date,
     session.country,
     session.device,
     session.continent,
     session.channel,
     session.test,
     session.test_group,
     'session' AS event_name,
     session_cnt AS value
FROM session
UNION ALL
SELECT
     account.date,
     account.country,
     account.device,
     account.continent,
     account.channel,
     account.test,
     account.test_group,
     'new account' AS event_name,
     new_account_cnt AS value
FROM account
"""

ab_test = client.query(query).to_dataframe()
ab_test.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-12-08,Palestine,desktop,Asia,Direct,4,2,new account,1
1,2020-12-08,Palestine,desktop,Asia,Direct,3,2,new account,1
2,2020-11-06,Puerto Rico,desktop,Americas,Social Search,2,2,new account,1
3,2020-11-06,Puerto Rico,desktop,Americas,Social Search,1,1,new account,1
4,2020-12-08,Croatia,desktop,Europe,Direct,4,2,new account,1


In [14]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm

In [15]:
# Event names
ab_test["event_name"].unique()

array(['new account', 'session with orders', 'scroll', 'user_engagement',
       'first_visit', 'session_start', 'view_item', 'page_view',
       'view_promotion', 'view_search_results', 'add_shipping_info',
       'add_to_cart', 'begin_checkout', 'select_promotion', 'select_item',
       'add_payment_info', 'click', 'session', 'view_item_list'],
      dtype=object)

In [16]:
# Defining metrics
key_metrics = ["add_payment_info", "add_shipping_info", "begin_checkout", "new account"]
denominator = "session"

filtered_data = ab_test[ab_test["event_name"].isin(key_metrics + [denominator])].copy()
filtered_data.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-12-08,Palestine,desktop,Asia,Direct,4,2,new account,1
1,2020-12-08,Palestine,desktop,Asia,Direct,3,2,new account,1
2,2020-11-06,Puerto Rico,desktop,Americas,Social Search,2,2,new account,1
3,2020-11-06,Puerto Rico,desktop,Americas,Social Search,1,1,new account,1
4,2020-12-08,Croatia,desktop,Europe,Direct,4,2,new account,1


In [17]:
def ab_test_total_tool(data):
    result = []

    # Get unique test numbers to process multiple tests if available
    tests = data["test"].unique()
    for test_number in tests:
        # Filter data for the current test
        test = data[data["test"] == test_number]

        # Identify which key funnel metrics are present in the current test data
        metrics = [metric for metric in test["event_name"].unique() if metric in key_metrics]

        for metric in metrics:
            # Separate data into Control (group 1) and Test (group 2)
            control_group = test[test["test_group"] == 1]
            test_group = test[test["test_group"] == 2]

            # Aggregate numerators (target events) and denominators (total sessions) for both groups
            control_event = control_group[control_group["event_name"] == metric]["value"].sum()
            control_denominator = control_group[control_group["event_name"] == denominator]["value"].sum()

            test_event = test_group[test_group["event_name"] == metric]["value"].sum()
            test_denominator = test_group[test_group["event_name"] == denominator]["value"].sum()

            # Calculate conversion rates (CR) for Control and Test groups
            if control_denominator > 0:
                control_group_conversion = control_event / control_denominator * 100
            else:
                control_group_conversion = 0

            if test_denominator > 0:
                test_group_conversion = test_event / test_denominator * 100
            else:
                test_group_conversion = 0

            # Calculate the relative percentage change in conversion rate
            if control_group_conversion > 0:
                metric_change = (test_group_conversion - control_group_conversion) / control_group_conversion * 100
            else:
                metric_change = 0

            # Perform a two-sample Z-test for proportions to determine statistical significance
            if control_denominator > 0 and test_denominator > 0:
                z_stat, p_value = sm.stats.proportions_ztest([control_event, test_event], [control_denominator, test_denominator])
                significant = p_value < 0.05  # True if statistically significant at alpha = 0.05
            else:
                z_stat, p_value, significant = None, None, None

            # Append all calculated metrics and statistical test results for the current row
            result.append({
                "test_number": test_number,
                "metric": f"{metric}/{denominator}",
                "numerator_event": metric,
                "denominator_event": denominator,
                "numerator_control": control_event,
                "denominator_control": control_denominator,
                "conversion_rate_control": control_group_conversion,
                "numerator_test": test_event,
                "denominator_test": test_denominator,
                "conversion_rate_test": test_group_conversion,
                "metric_change": metric_change,
                "z_stat": z_stat,
                "p_value": p_value,
                "significant": significant
            })

    # Convert the list of dictionaries into a clean, structured pandas DataFrame
    return pd.DataFrame(result)

ab_test_results = ab_test_total_tool(ab_test)
display(ab_test_results)

,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change,z_stat,p_value,significant
0,4,new account/session,new account,session,8984,105079,8.549758,8687,105141,8.262238,-3.362896,2.375457,0.017527,True
1,4,begin_checkout/session,begin_checkout,session,12555,105079,11.948153,12267,105141,11.667190,-2.351523,1.995998,0.045934,True
2,4,add_shipping_info/session,add_shipping_info,session,5128,105079,4.880138,4956,105141,4.713670,-3.411125,1.785795,0.074132,False
3,4,add_payment_info/session,add_payment_info,session,3731,105079,3.550662,3601,105141,3.424925,-3.541234,1.571106,0.116158,False
4,3,new account/session,new account,session,5856,70047,8.360101,5822,70439,8.265308,-1.133880,0.643489,0.519907,False
5,3,begin_checkout/session,begin_checkout,session,9532,70047,13.608006,9264,70439,13.151805,-3.352445,2.511389,0.012026,True
6,3,add_shipping_info/session,add_shipping_info,session,5298,70047,7.563493,5188,70439,7.365238,-2.621211,1.413727,0.157442,False
7,3,add_payment_info/session,add_payment_info,session,3623,70047,5.172241,3697,70439,5.248513,1.474630,-0.643172,0.520112,False
8,2,new account/session,new account,session,4165,50637,8.225211,4184,50244,8.327362,1.241934,-0.588793,0.556000,False
9,2,add_shipping_info/session,add_shipping_info,session,3480,50637,6.872445,3510,50244,6.985909,1.650995,-0.709557,0.477979,False


In [19]:
# Saving data for visualization in Tableau
from google.colab import drive
drive.mount("/content/drive")

ab_test_results.to_csv("/content/drive/MyDrive/Mate/ab_test_results.csv", index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


###**Vizualization in Tableau**
[Link to Dashboard](https://public.tableau.com/app/profile/anton.krokhin/viz/A_BTestingTool_17817954362310/ABtest?publish=yes)